# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Loading dataset

In [17]:
import os
from huggingface_hub import login
from datasets import load_dataset
from datetime import date

token = os.getenv("HF_TOKEN")
login(token)

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=token
)

# Converting to a pandas DataFrame
df = dataset["train"].to_pandas()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Expected unit of analysis:** One row represents the daily performance metrics of a single content page (`content_hash_id`) for a specific client (`client_hash_id`) on a single reporting date (`report_date`).

**Time window:** Daily (one record per content page, per client, per day).

**Verification:** This expected unit of analysis is verified in Section 3 using a mid-panel month (March 2026).

In [18]:
march_df = df[
    (df["report_date"] >= date(2026, 3, 1)) &
    (df["report_date"] <= date(2026, 3, 31))
]

print("March 2026 subset created")
print("Rows:", len(march_df))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

March 2026 subset created
Rows: 9841378
Date range: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

### Features
- `gsc_impressions` – Measures search visibility.
- `gsc_clicks` – Measures search traffic.
- `gsc_avg_position` – Indicates search ranking.
- `ga4_pageviews` – Measures page engagement.
- `ga4_engaged_sessions` – Measures user engagement quality.

### Label
- Refresh opportunity score (derived from search performance decline and engagement metrics to rank pages that should be refreshed).

### Context
- `report_date` – Used for filtering by time.
- `client_hash_id` – Identifies the client.
- `content_hash_id` – Identifies the content page.
- `gsc_data_available` – Used to verify Search Console data exists.
- `ga4_data_available` – Used to verify GA4 data exists.

### Excluded
- AI traffic source columns (`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`) because they are not directly relevant to identifying refresh opportunities in this assignment.

In [19]:
for col in march_df.columns:
    print(col)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events


### Label
- Refresh opportunity (target to be derived later from historical performance trends and engagement metrics).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Verify the unit of analysis

This query checks whether each `(report_date, client_hash_id, content_hash_id)` combination appears only once in the dataset.

In [20]:
duplicates = march_df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Total rows:", len(march_df))
print("Duplicate records:", duplicates)

if duplicates == 0:
    print("Grain verified: one row per content page, client and day.")
else:
    print("Found duplicate keys.")

Total rows: 9841378
Duplicate records: 0
Grain verified: one row per content page, client and day.


### Query 2: Verify the reporting window

This query confirms the size of the dataset and the date range covered by the warehouse table.

In [21]:
print("Row count:", len(march_df))
print("Start date:", march_df["report_date"].min())
print("End date:", march_df["report_date"].max())
print("Number of reporting dates:", march_df["report_date"].nunique())

Row count: 9841378
Start date: 2026-03-01
End date: 2026-03-31
Number of reporting dates: 31


### Query 3: Verify data availability

This query checks how many records have both Google Search Console and Google Analytics 4 data available.

In [22]:
available = march_df[
    (march_df["gsc_data_available"] == True) &
    (march_df["ga4_data_available"] == True)
]

print("Rows with both GSC and GA4 available:", len(available))

Rows with both GSC and GA4 available: 364347


### Feature frame

The following feature frame contains five historical features that can be used for refresh opportunity scoring.

In [23]:
feature_df = march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_engaged_sessions",
    ]
].copy()

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
35188099,20.0,0.0,3.350000,NaN,NaN
35188100,1.0,0.0,0.000000,NaN,NaN
35188101,125.0,1.0,4.928000,NaN,NaN
35188102,7.0,0.0,4.000000,NaN,NaN
35188103,11.0,0.0,2.272727,NaN,NaN


### Feature availability

- **gsc_impressions** – Knowable at the decision moment because historical Search Console impressions are already available.
- **gsc_clicks** – Knowable because historical click counts exist before deciding whether to refresh content.
- **gsc_avg_position** – Knowable because historical search rankings are already observed.
- **ga4_pageviews** – Knowable because pageview history has already been collected.
- **ga4_engaged_sessions** – Knowable because engagement metrics are historical observations available before the refresh decision.

### Leakage experiment

To demonstrate target leakage, a label-derived column is intentionally created. This produces an unrealistically strong predictor because it uses information that would not be available at prediction time. The column is then removed.

In [24]:
print("Features before leakage:", feature_df.columns.tolist())

feature_df["leaky_label"] = (
    march_df["gsc_clicks"] == 0
).astype(int)

print("Features after adding leakage:", feature_df.columns.tolist())

feature_df.drop(columns=["leaky_label"], inplace=True)

print("Features after removing leakage:", feature_df.columns.tolist())

Features before leakage: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']
Features after adding leakage: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions', 'leaky_label']
Features after removing leakage: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


## 4. Data limits

### Limitation 1: Incomplete data availability
Some rows do not have both Google Search Console (GSC) and Google Analytics 4 (GA4) data available. Analyses that require both sources must exclude these rows, which may reduce the amount of usable data.

### Limitation 2: Duplicate records
The verification step found a small number of duplicate (`report_date`, `client_hash_id`, `content_hash_id`) combinations. These duplicate records may slightly affect aggregations if they are not handled carefully.

### Limitation 3: Historical window constraints
The dataset contains historical snapshots, but it cannot explain *why* a page's performance changed. External factors such as Google algorithm updates, seasonality, competitor actions, or content quality are not captured directly.

### Limitation 4: Refresh opportunity is inferred
The dataset does not contain a ground-truth "refresh needed" label. Any refresh opportunity score must be derived from historical performance trends and engagement metrics rather than observed directly.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.